# Transformer vs LSTM 訓練 (Multi30k 独→英)

In [ ]:
!git clone https://github.com/mukai112358/transformer.git
%cd transformer

In [ ]:
!pip install -q datasets sacrebleu

In [ ]:
import os, sys, pickle
import torch

sys.path.insert(0, os.getcwd())
os.makedirs('results', exist_ok=True)

from src.data import get_dataloaders
from src.transformer import Transformer
from src.lstm_baseline import LSTMSeq2Seq
from src.train import run_training
from src.evaluate import translate_dataset

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

In [ ]:
loaders = get_dataloaders(batch_size=64, num_workers=2)
src_vocab, tgt_vocab = loaders['src_vocab'], loaders['tgt_vocab']
print('src vocab:', len(src_vocab), 'tgt vocab:', len(tgt_vocab))

In [ ]:
EPOCHS = 20
LR = 5e-4
D_MODEL = 256
NUM_HEADS = 8
D_FF = 1024
NUM_LAYERS = 3
MAX_LEN = 50
DROPOUT = 0.1

## Transformer 訓練

In [ ]:
torch.manual_seed(42)
transformer = Transformer(
    src_vocab_size=len(src_vocab), tgt_vocab_size=len(tgt_vocab),
    d_model=D_MODEL, num_heads=NUM_HEADS, d_ff=D_FF,
    num_layers=NUM_LAYERS, max_len=MAX_LEN, dropout=DROPOUT,
)
print(f'Transformer params: {sum(p.numel() for p in transformer.parameters()):,}')

history_t = run_training(transformer, loaders['train_loader'], loaders['val_loader'],
                         epochs=EPOCHS, lr=LR, device=device)

torch.save(transformer.state_dict(), 'results/transformer.pt')
with open('results/history_transformer.pkl', 'wb') as f:
    pickle.dump(history_t, f)

## LSTM 訓練

In [ ]:
torch.manual_seed(42)
lstm = LSTMSeq2Seq(
    src_vocab_size=len(src_vocab), tgt_vocab_size=len(tgt_vocab),
    embed_dim=D_MODEL, hidden_dim=320, num_layers=NUM_LAYERS, dropout=DROPOUT,
)
print(f'LSTM params: {sum(p.numel() for p in lstm.parameters()):,}')

history_l = run_training(lstm, loaders['train_loader'], loaders['val_loader'],
                         epochs=EPOCHS, lr=LR, device=device)

torch.save(lstm.state_dict(), 'results/lstm.pt')
with open('results/history_lstm.pkl', 'wb') as f:
    pickle.dump(history_l, f)

## 訳例の確認と保存

In [ ]:
hyps_t, refs, src_lens = translate_dataset(transformer, loaders['test_loader'], tgt_vocab, device)
hyps_l, _, _ = translate_dataset(lstm, loaders['test_loader'], tgt_vocab, device)

for i in range(5):
    print(f'--- sample {i} ---')
    print('REF :', refs[i])
    print('TRA :', hyps_t[i])
    print('LSTM:', hyps_l[i])
    print()

with open('results/translations_test.pkl', 'wb') as f:
    pickle.dump({'refs': refs, 'transformer': hyps_t, 'lstm': hyps_l, 'src_lens': src_lens}, f)